In [1]:
from pathlib import *
import os
import random
random.seed(42)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import json

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

with open('IDS/data/utils/mapping.json','r') as f:
        mapping = json.load(f)

with open('IDS/data/utils/filters.json','r') as f:
        filters = json.load(f)

In [ ]:
data_dirs = ["DDoS Evaluation Dataset (CIC-DDoS2019)/01-12"] # , "DDoS Evaluation Dataset (CIC-DDoS2019)/03-11"]
f_list = []
count_dict = {}
for data_dir in data_dirs:
    for f_path in tqdm(Path(data_dir).iterdir(), desc='collecting data...'):
        if os.path.isfile(f_path):
            df = pd.read_csv(f_path, usecols=[" Label"])
            df.rename(columns=lambda x: x.strip(), inplace=True)
            grouped = df.groupby('Label')
            for name, group in grouped:
                count_dict[name] = count_dict.setdefault(name,0) + len(group)
print(count_dict)

collecting data...: 11it [02:32, 13.86s/it]

{'BENIGN': 56863, 'DrDoS_DNS': 5071011, 'DrDoS_LDAP': 2179930, 'DrDoS_MSSQL': 4522492, 'DrDoS_NetBIOS': 4093279, 'DrDoS_NTP': 1202642, 'DrDoS_SNMP': 5159870, 'DrDoS_SSDP': 2610611, 'DrDoS_UDP': 3134645, 'Syn': 1582289, 'TFTP': 20082580, 'UDP-lag': 366461, 'WebDDoS': 439}


In [ ]:
sum(count_dict.values())

50063112

In [2]:
data_dirs = ["DDoS Evaluation Dataset (CIC-DDoS2019)/01-12"]

dead_cols = ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count',
       'PSH Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

#drop na, infs, drop duplicates

for data_dir in data_dirs:
    for f_path in tqdm(Path(data_dir).iterdir(), desc='collecting data...'):
        if os.path.isfile(f_path):
            df = pd.read_csv(f_path, dtype={'SimillarHTTP':str})
            df = df.rename(columns=lambda x: x.strip())
            df = df.drop(columns='Unnamed: 0')
            df['Label'] = df['Label'].map(mapping)
            df = df.select_dtypes(include=['int', 'float', 'bool'])
            df = df.drop(columns=dead_cols)
            df = df.replace([np.inf, -np.inf], np.nan)
            df = df.dropna()
            df = df.drop_duplicates()
            grouped = df.groupby('Label')
            for name, group in grouped:
                print(name, group.shape)
                if os.path.isfile(Path(f'IDS/data/Processed Data 01-12 ONLY/{name}.csv')):
                    group.to_csv(f'IDS/data/Processed Data 01-12 ONLY/{name}.csv', mode='a', index=False, header=False)
                else:
                    group.to_csv(f'IDS/data/Processed Data 01-12 ONLY/{name}.csv',index=False)

collecting data...: 0it [00:00, ?it/s]

0 (3223, 70)
1 (4773092, 70)


collecting data...: 1it [03:28, 208.95s/it]

0 (1472, 70)
2 (2104005, 70)


collecting data...: 2it [04:59, 139.27s/it]

...

0 (24859, 70)
10 (17859895, 70)


collecting data...: 10it [37:41, 395.38s/it]

0 (3684, 70)
11 (330070, 70)


collecting data...: 11it [37:57, 207.00s/it]

13 (439, 70)


In [ ]:
df = pd.read_csv('IDS/data/Processed Data 01-12 ONLY/WebDDoS.csv')
df['Label'] = 12
df.to_csv('IDS/data/Processed Data 01-12 ONLY/WebDDoS.csv', index=False)

In [3]:
data_dir = "IDS/data/Processed Data 01-12 ONLY/"
f_list = []
for f_path in tqdm(Path(data_dir).iterdir(), desc='collecting data...'):
    if os.path.isfile(f_path):
        f_list.append(f_path)
f_list

collecting data...: 13it [00:00, 3248.88it/s]


[WindowsPath('IDS/data/Processed Data 01-12 ONLY/Benign.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/DNS.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/LDAP.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/MSSQL.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/NetBIOS.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/NTP.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/SNMP.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/SSDP.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/Syn.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/TFTP.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/UDP.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/UDPLag.csv'),
 WindowsPath('IDS/data/Processed Data 01-12 ONLY/WebDDoS.csv')]

In [2]:
data_dir = "IDS/data/Processed Data/"
f_list = []
for f_path in tqdm(Path(data_dir).iterdir(), desc='collecting data...'):
    if os.path.isfile(f_path):
        f_list.append(f_path)
f_list

collecting data...: 14it [00:00, 13997.68it/s]


[WindowsPath('IDS/data/Processed Data/Benign.csv'),
 WindowsPath('IDS/data/Processed Data/DNS.csv'),
 WindowsPath('IDS/data/Processed Data/LDAP.csv'),
 WindowsPath('IDS/data/Processed Data/MSSQL.csv'),
 WindowsPath('IDS/data/Processed Data/NetBIOS.csv'),
 WindowsPath('IDS/data/Processed Data/NTP.csv'),
 WindowsPath('IDS/data/Processed Data/Portmap.csv'),
 WindowsPath('IDS/data/Processed Data/SNMP.csv'),
 WindowsPath('IDS/data/Processed Data/SSDP.csv'),
 WindowsPath('IDS/data/Processed Data/Syn.csv'),
 WindowsPath('IDS/data/Processed Data/TFTP.csv'),
 WindowsPath('IDS/data/Processed Data/UDP.csv'),
 WindowsPath('IDS/data/Processed Data/UDPLag.csv')]

In [10]:
for f_path in f_list:
    df = pd.read_csv(f_path)
    print(f_path)
    print(df.columns[df.eq(0).all()].to_list())
    print(df.columns[df.isna().all()].to_list())
    print('=============================')

IDS\data\Processed Data\Benign.csv
Index(['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count',
       'PSH Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Label'],
      dtype='object')
Index([], dtype='object')
IDS\data\Processed Data\DNS.csv
Index(['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count',
       'PSH Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'],
      dtype='object')
Index([], dtype='object')
IDS\data\Processed Data\LDAP.csv
Index(['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count',
       'PSH Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk

In [12]:
dead_cols = ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count',
       'PSH Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

for f_path in f_list:
    df = pd.read_csv(f_path, usecols=lambda column: column not in dead_cols)
    print(f_path, df.shape)
    df.to_csv(f_path, index=False)

IDS\data\Processed Data\Benign.csv (113828, 70)
IDS\data\Processed Data\DNS.csv (5071011, 70)
IDS\data\Processed Data\LDAP.csv (4095052, 70)
IDS\data\Processed Data\MSSQL.csv (10309945, 70)
IDS\data\Processed Data\NetBIOS.csv (7750776, 70)
IDS\data\Processed Data\NTP.csv (1202642, 70)
IDS\data\Processed Data\Portmap.csv (186960, 70)
IDS\data\Processed Data\SNMP.csv (5159870, 70)
IDS\data\Processed Data\SSDP.csv (2610611, 70)
IDS\data\Processed Data\Syn.csv (6473789, 70)
IDS\data\Processed Data\TFTP.csv (20082580, 70)
IDS\data\Processed Data\UDP.csv (7001800, 70)
IDS\data\Processed Data\UDPLag.csv (368334, 70)
IDS\data\Processed Data\WebDDoS.csv (439, 70)


In [14]:
for f_path in f_list:
    df = pd.read_csv(f_path)
    init_shape = df.shape[0]
    df = df.dropna()
    print(f_path, init_shape, df.shape[0], init_shape - df.shape[0])
    df.to_csv(f_path, index=False)

IDS\data\Processed Data\Benign.csv 113828 113673 155
IDS\data\Processed Data\DNS.csv 5071011 5071002 9
IDS\data\Processed Data\LDAP.csv 4095052 4095050 2
IDS\data\Processed Data\MSSQL.csv 10309945 10309938 7
IDS\data\Processed Data\NetBIOS.csv 7750776 7750765 11
IDS\data\Processed Data\NTP.csv 1202642 1202639 3
IDS\data\Processed Data\Portmap.csv 186960 186960 0
IDS\data\Processed Data\SNMP.csv 5159870 5159863 7
IDS\data\Processed Data\SSDP.csv 2610611 2610610 1
IDS\data\Processed Data\Syn.csv 6473789 6271438 202351
IDS\data\Processed Data\TFTP.csv 20082580 20072108 10472
IDS\data\Processed Data\UDP.csv 7001800 7001798 2
IDS\data\Processed Data\UDPLag.csv 368334 332202 36132
IDS\data\Processed Data\WebDDoS.csv 439 439 0


In [3]:
for f_path in f_list:
    df = pd.read_csv(f_path)
    init_shape = df.shape[0]
    df = df.drop_duplicates()
    df.reset_index()
    print(f_path, init_shape, df.shape[0], init_shape - df.shape[0])
    df.to_csv(f_path, index=False)

IDS\data\Processed Data\Benign.csv 113673 108119 5554
IDS\data\Processed Data\DNS.csv 5071002 4935081 135921
IDS\data\Processed Data\LDAP.csv 4095050 3958370 136680
IDS\data\Processed Data\MSSQL.csv 10309938 10301075 8863
IDS\data\Processed Data\NetBIOS.csv 7750765 7356948 393817
IDS\data\Processed Data\NTP.csv 1202639 1202515 124
IDS\data\Processed Data\Portmap.csv 186960 186957 3
IDS\data\Processed Data\SNMP.csv 5159863 5048212 111651
IDS\data\Processed Data\SSDP.csv 2610610 2610579 31
IDS\data\Processed Data\Syn.csv 6271438 5790952 480486
IDS\data\Processed Data\TFTP.csv 20072108 18400026 1672082
IDS\data\Processed Data\UDP.csv 7001798 7001644 154
IDS\data\Processed Data\UDPLag.csv 332202 332193 9
IDS\data\Processed Data\WebDDoS.csv 439 439 0


In [10]:
for f_path in f_list:
    df = pd.read_csv(f_path)
    init_shape = df.shape[0]
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df = df.dropna()
    print(f_path, init_shape, df.shape[0], init_shape - df.shape[0])
    df.to_csv(f_path, index=False)

IDS\data\Processed Data\Benign.csv 108119 107230 889
IDS\data\Processed Data\DNS.csv 4935081 4773092 161989
IDS\data\Processed Data\LDAP.csv 3958370 3876860 81510
IDS\data\Processed Data\MSSQL.csv 10301075 9971677 329398
IDS\data\Processed Data\NetBIOS.csv 7356948 7087440 269508
IDS\data\Processed Data\NTP.csv 1202515 1195569 6946
IDS\data\Processed Data\Portmap.csv 186957 177194 9763
IDS\data\Processed Data\SNMP.csv 5048212 5037610 10602
IDS\data\Processed Data\SSDP.csv 2610579 2568538 42041
IDS\data\Processed Data\Syn.csv 5790952 5496320 294632
IDS\data\Processed Data\TFTP.csv 18400026 17859895 540131
IDS\data\Processed Data\UDP.csv 7001644 6884146 117498
IDS\data\Processed Data\UDPLag.csv 332193 331943 250


In [4]:
for f_path in f_list:
    df = pd.read_csv(f_path)
    print(f_path)
    print(df.columns[df.eq(0).all()].to_list())
    print(df.columns[df.isna().all()].to_list())
    print('=============================')

IDS\data\Processed Data\Benign.csv
['Label']
[]
IDS\data\Processed Data\DNS.csv
[]
[]
IDS\data\Processed Data\LDAP.csv
[]
[]
IDS\data\Processed Data\MSSQL.csv
[]
[]
IDS\data\Processed Data\NetBIOS.csv
[]
[]
IDS\data\Processed Data\NTP.csv
[]
[]
IDS\data\Processed Data\Portmap.csv
[]
[]
IDS\data\Processed Data\SNMP.csv
[]
[]
IDS\data\Processed Data\SSDP.csv
[]
[]
IDS\data\Processed Data\Syn.csv
[]
[]
IDS\data\Processed Data\TFTP.csv
[]
[]
IDS\data\Processed Data\UDP.csv
[]
[]
IDS\data\Processed Data\UDPLag.csv
[]
[]
IDS\data\Processed Data\WebDDoS.csv
['Fwd Packet Length Min', 'Bwd Packet Length Min', 'Min Packet Length', 'SYN Flag Count', 'Active Std', 'Idle Std']
[]


In [8]:
for f_path in f_list:
    df = pd.read_csv(f_path)
    print(f_path)
    print((df == 0).sum() / df.shape[0])
    print('=============================')

IDS\data\Processed Data\Benign.csv
Source Port                    0.015076
Destination Port               0.015076
Protocol                       0.015076
Flow Duration                  0.008222
Total Fwd Packets              0.000000
Total Backward Packets         0.228507
Total Length of Fwd Packets    0.237109
Total Length of Bwd Packets    0.422146
Fwd Packet Length Max          0.237109
Fwd Packet Length Min          0.486843
Fwd Packet Length Mean         0.237109
Fwd Packet Length Std          0.689268
Bwd Packet Length Max          0.422146
Bwd Packet Length Min          0.575865
Bwd Packet Length Mean         0.422146
Bwd Packet Length Std          0.829216
Flow Bytes/s                   0.223189
Flow Packets/s                 0.000000
Flow IAT Mean                  0.008222
Flow IAT Std                   0.168398
Flow IAT Max                   0.008222
Flow IAT Min                   0.080939
Fwd IAT Total                  0.179580
Fwd IAT Mean                   0.179580
Fwd I